# M04 — Física directa y controles de prismas inducidos

**Level 3 · Prerrequisitos: notebook de fundamentos; lectura: 35 minutos.**

## Objetivos

Ejecute el operador directo de prismas inducidos sobre el fixture sintético congelado. Compare componentes con el oráculo físico Choclo independiente y distinga TMI lineal de anomalía exacta de magnitud. No es inversión de susceptibilidad ni clasificador de remanencia.

## Preparación

Use el entorno pipeline Python 3.12 ya revisado. Este notebook no instala nada.

Importe el código del repositorio y confirme su archivo fuente.

In [ ]:
import runpy,sys
from pathlib import Path
import numpy as np
root=Path.cwd();sys.path.insert(0,str(root/'data-pipeline'))
import magnetic_forward
# Reuse the product's frozen synthetic physical control.
fixture=runpy.run_path(str(root/'tests/numerics/test_magnetic_forward.py'))
print(magnetic_forward.ENGINE)

## Datos

Ejecute el operador directo de prismas inducidos sobre el fixture sintético congelado. Compare componentes con el oráculo físico Choclo independiente y distinga TMI lineal de anomalía exacta de magnitud. No es inversión de susceptibilidad ni clasificador de remanencia.

**Files:** tests/numerics/test_magnetic_forward.py, docs/design/features/m04-induced-prism/algorithms.md, docs/design/features/m04-induced-prism/validation.md

## Método

Para I positiva hacia abajo y D horario desde norte, f=(cos I sin D, cos I cos D, −sin I) ENU; B₀=Ff. La magnetización inducida uniforme es M=χ·10⁻⁹B₀/μ₀ A m⁻¹. b secundario da TMI lineal f·b; la anomalía exacta es |B₀+b|−F. Supuestos: campo uniforme, susceptibilidad escalar, sin autodesmagnetización ni remanencia.

Ejecute el método del producto sobre el control declarado.

In [ ]:
request=fixture['request']()
# Compare actual product response and independent physical prism calculation.
result=magnetic_forward.forward_magnetic(request)
oracle=fixture['physical_oracle'](request)
max_error=float(np.max(np.abs(result['field_components_nt']-oracle)))
print({'componentes_nT':result['field_components_nt'].tolist(),'TMI_lineal_nT':result['linear_tmi_nt'].tolist(),'anomalia_magnitud_nT':result['exact_magnitude_anomaly_nt'].tolist(),'error_componente_maximo_nT':max_error,'aceptado':result['linearization']['full_method_accepted']})

Control adversarial/negativo: aísle un supuesto o incumpla una restricción física declarada.

In [ ]:
from copy import deepcopy
changed=deepcopy(request);changed['inducing_field']['declination_deg']=42.0
# Change only a declared field assumption; this is not field classification.
alternative=magnetic_forward.forward_magnetic(changed)
delta=float(np.max(np.abs(alternative['field_components_nt']-result['field_components_nt'])))
print({'max_component_change_nT':delta})
assert delta>0

## Estado del arte

SimPEG/Geoana es producción; Choclo usa otro cálculo analítico de prisma. Cambiar el campo prueba sensibilidad, no identifica el campo real. Magnitud exacta y TMI lineal difieren; el Jacobiano devuelto corresponde solo a TMI lineal.

## Resultados

Use tolerancias independientes congeladas, no valores redondeados. Cambiar declinación debe cambiar componentes. El operador deja falsos origen de campo y aceptación completa; la inversión M04 queda fuera de esta lección.

## Resumen

Mantenga juntas convención, unidades, observables y oráculo. Una imagen directa no es geología recuperada.

## Referencias

- https://raw.githubusercontent.com/simpeg/simpeg/v0.25.2/simpeg/potential_fields/magnetics/sources.py
- https://raw.githubusercontent.com/simpeg/simpeg/v0.25.2/simpeg/utils/mat_utils.py
- https://www.fatiando.org/choclo/v0.3.2/api/generated/choclo.prism.magnetic_field.html
- https://doi.org/10.1016/j.cageo.2015.09.015